# Generation Walkthrough

Notebook for batch sampling, VAE rendering, and direct inspection of captured arrays.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))

from client.config import registry
from client.interface import SlopClient

print(registry.list())


[ProviderConfig(name='vast-32582479', kind='ssh', target='vast-32582479', remote_path='/root/slop', python_cmd='python3', container_image=None, num_workers=8)]


In [ ]:
providers = registry.list()
assert providers, 'no providers registered'
cfg = providers[-1]
client = SlopClient(cfg)
client.connect()

In [ ]:
result = client.sample(
    prompt='a brass sphere on a wooden table',
    batch_size=3,
    num_steps=6,
)
result.points.shape, result.forces.shape

In [ ]:
final_points = result.points[-1]
decoded = client.render(final_points)
len(decoded)

for d in decoded:
    plt.figure(figsize=(8, 8))
    plt.imshow(d)
    plt.axis('off')
    plt.title('Generated Image')
    plt.show()

In [ ]:
force_norms = np.linalg.norm(result.forces.reshape(result.forces.shape[0], result.forces.shape[1], -1), axis=-1)
force_norms.shape

In [ ]:
plt.figure(figsize=(8, 4))
for i in range(force_norms.shape[1]):
    plt.plot(force_norms[:, i], label=f'batch {i}')
plt.title('Force Norm By Step')
plt.xlabel('Step')
plt.ylabel('Norm')
plt.legend()
plt.show()

In [ ]:
client.close()